In [ ]:
# ============================================================
# Colab: Mann–Kendall (PC1) por município + estatísticas (mean/median/std)
# Saída: projects/rnvuefsppgm/assets/Luana_RSEI_TRENDS
# ============================================================

# -*- coding: utf-8 -*-

import ee
import os
import sys
import copy
import re

# Initialize a map object.
import ee
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
# ee.Initialize(project=os.environ.get('EE_PROJECT'))
ee.Initialize(project=os.environ.get('EE_PROJECT'))
import geemap.core as geemap
# Cloud authentication.
from google.colab import auth
auth.authenticate_user()

In [ ]:
# 3) PARÂMETROS DO USUÁRIO
asset_indices = 'projects/nexgenmap/MapBiomas2/LANDSAT/DEGRADACAO/mosaics-ndfia'
asset_city    = 'projects/ee-rnvuefsppgm/assets/CAP_JOIN2'

# Coleção de saída (ImageCollection)
#asset_out_root = 'projects/rnvuefsppgm/assets/Luana_RSEI_TRENDS_GAP'
asset_out_root = 'projects/rnvuefsppgm/assets/Luana_RSEI_TRENDS'
# Lista de capitais
MUNICIPIOS = [
  #'Porto Velho','Rio Branco','Manaus','Boa Vista','Belém','Macapá','Palmas','São Luís','Teresina',
  #'Fortaleza','Natal','João Pessoa','Recife','Maceió','Aracaju',
  'Salvador'
  ##,'Belo Horizonte',
  #'Vitória','Rio de Janeiro','São Paulo','Curitiba','Florianópolis','Porto Alegre','Campo Grande',
  #'Cuiabá','Goiânia',
  #'Brasília'
]

In [ ]:
# 4) Utilitários
import re
import ee

def sanitize_id(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]', '_', name)

def ensure_image_collection(ic_asset_id: str):
    """Garante que exista uma ImageCollection no asset indicado."""
    try:

        info = ee.data.getAsset(ic_asset_id)
        if info and info.get('type') == 'IMAGE_COLLECTION':
            return  # já existe
        elif info:
            print(f"[AVISO] Asset existe mas não é ImageCollection: {ic_asset_id} (type={info.get('type')}).")
            return
    except Exception:
        pass
    try:
        ee.data.createAsset({'type': ee.data.ASSET_TYPE_IMAGE_COLL}, ic_asset_id)
        print(f"[OK] ImageCollection criada: {ic_asset_id}")
    except Exception as e:
        print(f"[ERRO] Falha ao criar ImageCollection {ic_asset_id}: {e}")

# =================== NOVO: PREENCHE LACUNAS TEMPORAIS (PC1) ===================
def temporal_gapfill_ic_bidirectional(ic, band_name='PC1'):
    """
    Preenche pixels mascarados (NoData) em cada imagem da coleção usando informação temporal.
    Estratégia em 2 passos:
      1) Backward (usa t+1): ordena DESC por tempo e faz unmask com o acumulador da próxima imagem.
      2) Forward  (usa t-1): ordena ASC  por tempo e faz unmask com o acumulador da imagem anterior.
    Retorna a coleção preenchida em ordem temporal ASC (antigo→recente), preservando system:time_start.
    """
    # Garante apenas a banda de interesse (por segurança)
    ic = ic.select([band_name])

    # --- Passo 1: Backward fill (usa valores do futuro: t+1) ---
    ic_asc  = ic.sort('system:time_start')
    size    = ic_asc.size()
    ic_desc = ic_asc.sort('system:time_start', False)
    list_desc = ic_desc.toList(size)

    # acumulador inicial: imagem totalmente mascarada (não altera unmask)
    init_acc = ee.Dictionary({
        'prev': ee.Image(0).updateMask(ee.Image(0)),
        'list': ee.List([])
    })

    def _step_backward(el, acc):
        acc = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        # Preenche os buracos de 'img' com valores de 'prev' (que carrega t+1, t+2, ...)
        filled = img.unmask(prev)
        # Preserva propriedades/tempo da imagem original
        filled = filled.copyProperties(img, img.propertyNames())
        return ee.Dictionary({
            'prev': filled,
            'list': ee.List(acc.get('list')).add(filled)
        })

    back_dict = ee.Dictionary(ee.List(list_desc).iterate(_step_backward, init_acc))
    # Lista preenchida em ordem DESC; reverte para ASC
    list_back_asc = ee.List(back_dict.get('list')).reverse()
    ic_back_asc = ee.ImageCollection.fromImages(list_back_asc)

    # --- Passo 2: Forward fill (usa valores do passado: t-1) ---
    list_fwd = ic_back_asc.toList(size)

    init_acc_fwd = ee.Dictionary({
        'prev': ee.Image(0).updateMask(ee.Image(0)),
        'list': ee.List([])
    })

    def _step_forward(el, acc):
        acc = ee.Dictionary(acc)
        prev = ee.Image(acc.get('prev'))
        img  = ee.Image(el)
        # Preenche os buracos de 'img' com valores de 'prev' (que carrega t-1, t-2, ...)
        filled = img.unmask(prev)
        filled = filled.copyProperties(img, img.propertyNames())
        return ee.Dictionary({
            'prev': filled,
            'list': ee.List(acc.get('list')).add(filled)
        })

    fwd_dict = ee.Dictionary(ee.List(list_fwd).iterate(_step_forward, init_acc_fwd))
    list_filled = ee.List(fwd_dict.get('list'))
    ic_filled = ee.ImageCollection.fromImages(list_filled)

    # Garante nome da banda e ordem temporal final
    return ic_filled.sort('system:time_start').select([band_name])


In [ ]:
# 5) Funções do pipeline
def mk_for_municipio(municipio: str):
    """
    Constrói imagem com bandas:
      - significant (0/1), Z, S (Mann–Kendall sobre PC1)
      - mean, median, stdDev (estatísticas temporais da PC1)
    e retorna dict com info + task de export para Asset.
    """
    # Geometria do município
    fc_city = ee.FeatureCollection(asset_city).filter(ee.Filter.eq('NM_MUN', municipio))
    geom_city = fc_city.geometry()

    # ImageCollection filtrada por município e ordenada no tempo
    ic_raw = ee.ImageCollection(asset_indices) \
                .filter(ee.Filter.eq('NM_MUN', municipio)) \
                .sort('system:time_start')

    # Checa número de imagens no cliente
    try:
        n_info = ic_raw.size().getInfo()
    except Exception as e:
        return {'municipio': municipio, 'status': 'erro', 'msg': f'Não foi possível obter tamanho da IC: {e}'}

    if not n_info or n_info < 2:
        return {'municipio': municipio, 'status': 'pulado', 'msg': f'Série com {n_info} imagem(ns); precisa de >= 2.'}

    # Garante que a banda PC1 existe
    try:
        bands0 = ee.Image(ic_raw.first()).bandNames().getInfo()
        if 'PC1' not in bands0:
            return {'municipio': municipio, 'status': 'erro', 'msg': "Banda 'PC1' não encontrada na coleção."}
    except Exception as e:
        return {'municipio': municipio, 'status': 'erro', 'msg': f'Falha ao checar bandas: {e}'}

    # Mantém apenas PC1 e recorta ao município
    ic = ic_raw.select(['PC1']).map(lambda img: ee.Image(img).clip(geom_city))

    # ========= NOVO: preenche vazios temporais (t+1 e t-1) ANTES do trend =========
    ic = temporal_gapfill_ic_bidirectional(ic, 'PC1')

    # ===== Mann–Kendall sobre PC1 =====
    time_filter   = ee.Filter.lessThan(leftField='system:time_start', rightField='system:time_start')
    save_all_join = ee.Join.saveAll(matchesKey='after', ordering='system:time_start', ascending=True)
    joined        = save_all_join.apply(ic, ic, time_filter)

    def sign(i, j):
        i = ee.Image(i)
        j = ee.Image(j)
        return j.neq(i).multiply(j.subtract(i).clamp(-1, 1)).int()

    kendall = ee.ImageCollection(
        joined.map(lambda current:
            ee.ImageCollection.fromImages(ee.List(ee.Image(current).get('after'))).map(
                lambda image: sign(ee.Image(current), ee.Image(image)).unmask(0)
            )
        ).flatten()
    ).reduce('sum', 2).rename('S').clip(geom_city)

    # Variância (aprox., sem correção por ties)
    n_num = ee.Number(n_info)
    kendall_variance = n_num.multiply(n_num.subtract(1)).multiply(n_num.multiply(2).add(5)).divide(18)

    zero = kendall.select('S').multiply(kendall.select('S').eq(0))
    pos  = kendall.select('S').multiply(kendall.select('S').gt(0)).subtract(1)
    neg  = kendall.select('S').multiply(kendall.select('S').lt(0)).add(1)

    z = zero.add(pos.divide(kendall_variance.sqrt())).add(neg.divide(kendall_variance.sqrt())) \
             .rename('Z').clip(geom_city)

    # p-value bicaudal e significância (p <= 0.025)
    def ee_cdf(z_img):
        return ee.Image(0.5).multiply(ee.Image(1).add(ee.Image(z_img).divide(ee.Image(2).sqrt()).erf()))
    p = ee.Image(1).subtract(ee_cdf(z.abs())).rename('p').clip(geom_city)
    significant = p.lte(0.025).rename('significant').clip(geom_city)  # 0/1

    # ===== Estatísticas temporais (PC1) =====
    # mean, median, stdDev ao longo do tempo
    mean_img   = ic.reduce(ee.Reducer.mean()).rename(['mean']).clip(geom_city)
    median_img = ic.reduce(ee.Reducer.median()).rename(['median']).clip(geom_city)
    std_img    = ic.reduce(ee.Reducer.stdDev()).rename(['stdDev']).clip(geom_city)

    # Empacota as bandas na ordem desejada
    out_img = ee.Image.cat([
        significant.toByte(),  # 0/1
        z.toFloat(),           # Z
        kendall.toFloat(),     # S
        mean_img.toFloat(),    # mean(PC1)
        median_img.toFloat(),  # median(PC1)
        std_img.toFloat()      # stdDev(PC1)
    ]).rename(['significant', 'Z', 'S', 'mean', 'median', 'stdDev']).clip(geom_city)

    # Propriedades (primeira imagem da série bruta)
    src_img   = ee.Image(ic_raw.first())
    src_props = ee.List(src_img.propertyNames())

    # Datas inicial e final (da série PC1)
    start_date = ee.Date(ee.Image(ic.first()).get('system:time_start'))
    end_date   = ee.Date(ee.Image(ic.sort('system:time_start', False).first()).get('system:time_start'))
    m_pairs    = n_num.multiply(n_num.subtract(1)).divide(2)

    out_img = out_img.copyProperties(source=src_img, properties=src_props).set({
        'NM_MUN': municipio,
        'mk:first_band': 'PC1',
        'mk:n_images': n_num,
        'mk:p_threshold': 0.025,
        'mk:variance': kendall_variance,
        'mk:m_pairs': m_pairs,
        'mk:start': start_date.format('YYYY-MM-dd'),
        'mk:end': endDate.format('YYYY-MM-dd') if False else end_date.format('YYYY-MM-dd'),
        'mk:asset_indices': asset_indices,
        'mk:asset_city': asset_city,
        'mk:description': 'MK (S, Z, significant) + estatísticas temporais (mean/median/stdDev) da PC1 por município'
    })

    # Asset de saída
    muni_safe   = sanitize_id(municipio)
    asset_id_out = f"{asset_out_root}/MK_{muni_safe}"

    # Task de export
    task = ee.batch.Export.image.toAsset(
        image = out_img,
        description = f"MK_{muni_safe}",
        assetId = asset_id_out,
        region = geom_city,
        scale = 30,
        maxPixels = 1e13
    )

    return {
        'municipio': municipio,
        'status': 'ok',
        'n_images': n_info,
        'asset_out': asset_id_out,
        'task': task
    }

In [ ]:
# 6) Garante a ImageCollection de saída
ensure_image_collection(asset_out_root)

# 5) NORMALIZAÇÃO (tradução do JS para Python/EE)

def normalize_band_01(img, band_name, region, scale=30):
    """
    Normaliza (0-1) uma banda de uma Image, calculando min/max na região.
    Retorna dict: {'image': ee.Image, 'min': ee.Number, 'max': ee.Number}
    """
    stats = img.select([band_name]).reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    )
    statsD = ee.Dictionary(stats)
    # Valores seguros (defaults evitam erro quando tudo está mascarado)
    minv = ee.Number(statsD.get(f"{band_name}_min", 0))
    maxv = ee.Number(statsD.get(f"{band_name}_max", 1))
    rng  = maxv.subtract(minv)
    safe_rng = ee.Number(ee.Algorithms.If(rng.gt(0), rng, 1))

    minImg = ee.Image.constant(minv)
    safeRImg = ee.Image.constant(safe_rng)

    normBand = img.select([band_name]).subtract(minImg).divide(safeRImg).rename(band_name)
    # Mantém apenas o timestamp para preservar ordenação temporal
    normBand = normBand.copyProperties(img, ['system:time_start'])

    # Recompõe com as demais bandas (caso existam) e retorna
    others = img.bandNames().remove(band_name)
    out = ee.Image.cat(img.select(others), normBand)
    return {'image': out, 'min': minv, 'max': maxv}

def normalize_ic_01(ic, band_name, region, scale=30):
    """
    Normaliza (0-1) uma ImageCollection usando um min/max global da coleção (na região).
    Retorna dict: {'collection': ee.ImageCollection, 'min': ee.Number, 'max': ee.Number}
    """
    minImgT = ic.select(band_name).min()
    maxImgT = ic.select(band_name).max()

    minDict = minImgT.reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    )
    maxDict = maxImgT.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=region,
        scale=scale,
        maxPixels=1e13,
        bestEffort=True
    )

    minv = ee.Number(ee.Dictionary(minDict).get(band_name, 0))
    maxv = ee.Number(ee.Dictionary(maxDict).get(band_name, 1))
    rng  = maxv.subtract(minv)
    safe_rng = ee.Number(ee.Algorithms.If(rng.gt(0), rng, 1))

    minImg  = ee.Image.constant(minv)
    safeRImg = ee.Image.constant(safe_rng)

    def _map_norm(img):
        img = ee.Image(img)
        norm = img.select(band_name).subtract(minImg).divide(safeRImg).rename(band_name)
        # preserva apenas o timestamp para o join temporal
        norm = norm.copyProperties(img, ['system:time_start'])
        return norm

    ic_norm = ic.map(_map_norm)
    return {'collection': ic_norm, 'min': minv, 'max': maxv}

In [ ]:
# 6) Função principal (MK + estatísticas usando PC1 normalizada)
def mk_for_municipio(municipio: str):
    """
    Executa MK (S, Z, significant p<=0.025) e estatísticas (mean/median/stdDev)
    sobre a banda PC1 **normalizada (0-1)**. Exporta para Asset.
    Mantém apenas a propriedade 'NM_MUN' (string) na imagem final.
    """
    try:
        # Geometria do município
        fc_city = ee.FeatureCollection(asset_city).filter(ee.Filter.eq('NM_MUN', municipio))
        geom_city = fc_city.geometry()

        # IC filtrada por município e ordenada
        ic_raw = ee.ImageCollection(asset_indices) \
                    .filter(ee.Filter.eq('NM_MUN', municipio)) \
                    .sort('system:time_start')

        # tamanho mínimo
        n_info = ic_raw.size().getInfo()
        if not n_info or n_info < 2:
            return {'municipio': municipio, 'status': 'pulado', 'msg': f'{n_info} imagem(ns); precisa de >= 2.'}

        # checa banda PC1
        bands0 = ee.Image(ic_raw.first()).bandNames().getInfo()
        if 'PC1' not in bands0:
            return {'municipio': municipio, 'status': 'erro', 'msg': "Banda 'PC1' não encontrada na coleção."}

        # seleciona PC1 e recorta
        ic_pc1 = ic_raw.select(['PC1']).map(lambda img: ee.Image(img).clip(geom_city))

        # === NOVO: preenche temporalmente (t+1 e t-1) antes de normalizar/rodar MK ===
        ic_pc1_filled = temporal_gapfill_ic_bidirectional(ic_pc1, 'PC1')

        # === Normalização 0–1 na coleção inteira (mesmo min/max para todas as imagens)
        norm_res = normalize_ic_01(ic_pc1_filled, 'PC1', geom_city, scale=30)
        ic = ee.ImageCollection(norm_res['collection'])  # coleção normalizada (0–1)

        # ===== Mann–Kendall (S) =====
        time_filter   = ee.Filter.lessThan(leftField='system:time_start', rightField='system:time_start')
        save_all_join = ee.Join.saveAll(matchesKey='after', ordering='system:time_start', ascending=True)
        joined        = save_all_join.apply(ic, ic, time_filter)

        def sign(i, j):
            i = ee.Image(i)
            j = ee.Image(j)
            return j.neq(i).multiply(j.subtract(i).clamp(-1, 1)).int()

        kendall = ee.ImageCollection(
            joined.map(lambda current:
                ee.ImageCollection.fromImages(ee.List(ee.Image(current).get('after'))).map(
                    lambda image: sign(ee.Image(current), ee.Image(image)).unmask(0)
                )
            ).flatten()
        ).reduce('sum', 2).rename('S').clip(geom_city)

        # Variância (aprox., sem ties)
        n_num = ee.Number(n_info)
        kendall_variance = n_num.multiply(n_num.subtract(1)).multiply(n_num.multiply(2).add(5)).divide(18)

        zero = kendall.select('S').multiply(kendall.select('S').eq(0))
        pos  = kendall.select('S').multiply(kendall.select('S').gt(0)).subtract(1)
        neg  = kendall.select('S').multiply(kendall.select('S').lt(0)).add(1)

        z = zero.add(pos.divide(kendall_variance.sqrt())).add(neg.divide(kendall_variance.sqrt())) \
                 .rename('Z').clip(geom_city)

        # p e significância
        def ee_cdf(z_img):
            return ee.Image(0.5).multiply(ee.Image(1).add(ee.Image(z_img).divide(ee.Image(2).sqrt()).erf()))
        p = ee.Image(1).subtract(ee_cdf(z.abs())).rename('p').clip(geom_city)
        significant = p.lte(0.025).rename('significant').toByte().clip(geom_city)  # 0/1

        # ===== Estatísticas temporais sobre PC1 normalizada =====
        mean_img   = ic.reduce(ee.Reducer.mean()).rename(['mean']).toFloat().clip(geom_city)
        median_img = ic.reduce(ee.Reducer.median()).rename(['median']).toFloat().clip(geom_city)
        std_img    = ic.reduce(ee.Reducer.stdDev()).rename(['stdDev']).toFloat().clip(geom_city)

        # Empacota as 6 bandas e garante que todas são float
        out_img = ee.Image.cat([
            significant.toFloat(),         # 0/1 -> float
            z.toFloat(),         # Z -> float
            kendall.toFloat(),   # S -> float
            mean_img.toFloat(),            # mean(PC1_norm) -> float
            median_img.toFloat(),          # median(PC1_norm) -> float
            std_img.toFloat()              # stdDev(PC1_norm) -> float
        ]).rename(['significant','Z','S','mean','median','stdDev']).clip(geom_city)

        # Propriedade única
        out_img = out_img.set('NM_MUN', str(municipio))

        # Export
        muni_safe = sanitize_id(municipio)
        asset_id_out = f"{asset_out_root}/MK_{muni_safe}"

        task = ee.batch.Export.image.toAsset(
            image=out_img,
            description=f"MK_{muni_safe}",
            assetId=asset_id_out,
            region=geom_city,
            scale=30,
            maxPixels=1e13
        )

        return {'municipio': municipio, 'status': 'ok', 'asset_out': asset_id_out, 'task': task}

    except Exception as e:
        return {'municipio': municipio, 'status': 'erro', 'msg': str(e)}

In [ ]:
# 7) Garante a ImageCollection de saída
ensure_image_collection(asset_out_root)

# 8) Loop principal (cria e inicia tasks)
results = []
for m in MUNICIPIOS:
    info = mk_for_municipio(m)
    results.append(info)
    if info.get('status') == 'ok':
        print(f"[OK] Task criada: {info['municipio']} -> {info['asset_out']}")
        # The mk_for_municipio function returns a dictionary containing the task.
        # We need to access the task object from the dictionary to start it.
        if 'task' in info and info['task'] is not None:
          info['task'].start()
        else:
          print(f"[ERRO] Task não encontrada no resultado para {info['municipio']}")
    else:
        print(f"[{info.get('status','erro').upper()}] {m}: {info.get('msg','')}")

# 9) Resumo
ok_count = sum(1 for r in results if r.get('status') == 'ok')
skip_count = sum(1 for r in results if r.get('status') == 'pulado')
err_count = sum(1 for r in results if r.get('status') not in ('ok','pulado'))

print("\n==================== RESUMO ====================")
print(f"Tasks OK criadas: {ok_count}")
print(f"Pulados (<2 imgs): {skip_count}")
print(f"Erros: {err_count}")
for r in results:
    print(f" - {r['municipio']}: {r.get('status')} ({r.get('msg','')})")

print("\nDica: monitore as tasks com:")
print("for t in ee.batch.Task.list():\n    print(t.status())")